# Experimento 5 · Análisis de sentimiento de las reseñas con JEV (LLM)

> **Requiere la API key de JEV.** No necesita GPU: el modelo corre del lado del proveedor.

**Idea.** En lugar de un clasificador afinado (BETO), se le pide a un modelo de lenguaje grande que lea cada reseña y diga si el autor opinó positiva, neutral o negativamente de la película, más un puntaje continuo en [-1, 1]. Un LLM entiende sarcasmo, modismos mexicanos y emojis sin entrenamiento adicional (*zero-shot*), a cambio de costo por uso y menor reproducibilidad.

**Diseño de la consulta.**
* Se envían **lotes de 20 reseñas** por petición (menos peticiones y menos costo que una por una), numeradas; el modelo responde un arreglo JSON con la misma numeración.
* **Nunca** se le muestran las estrellas: solo el texto. Las estrellas se usan después para evaluar.
* Temperatura 0 para que la respuesta sea lo más estable posible.
* Todo lo recibido se guarda en un caché (`datos/limpios/sentimiento/jev_cache.jsonl`): si se interrumpe, al volver a correr retoma donde iba y no paga dos veces.

**Por seguridad, el notebook primero corre un piloto** de 100 reseñas, muestra su calidad y el consumo de tokens, y solo procesa todo si pones `EJECUTAR_TODO = True`.

**Entrada:** `datos/limpios/resenas_sentimiento.parquet` (columna `texto_llm`: limpia, conserva emojis).
**Salida:** `datos/limpios/sentimiento/sentimiento_jev.parquet`, con el mismo formato que BETO. Con ese archivo, los notebooks 01–03 corren la **variante C**.

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os; os.environ['TT_RAIZ'] = '/content/drive/MyDrive/TT_Recomendador_Cine_Mexicano'

import os, sys, warnings
from pathlib import Path
RAIZ = Path(os.environ.get('TT_RAIZ', Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
os.environ['TT_RAIZ'] = str(RAIZ)
sys.path.insert(0, str(RAIZ / 'src'))
warnings.filterwarnings('ignore', category=FutureWarning)
print('Raíz del proyecto:', RAIZ)

Raíz del proyecto: /content/drive/MyDrive/TT_Recomendador_Cine_Mexicano


In [4]:
# --- Credenciales y conexión -----------------------------------------------------
# La API key NO se escribe en el notebook. Defínela como variable de entorno
# (JEV_API_KEY) o pégala cuando se pida. En Colab también puedes usar Secretos.
import os
from getpass import getpass

JEV_API_KEY = os.environ.get('JEV_API_KEY') or getpass('API key de JEV: ')
JEV_BASE_URL = os.environ.get('JEV_BASE_URL', 'https://api.defapi.org/v1')   # ← ajustar
JEV_MODELO = os.environ.get('JEV_MODELO', 'openai/gpt-6-luna')          # ← ajustar

# --- Parámetros ---------------------------------------------------------------------
RESENAS_POR_PETICION = 20
HILOS = 4                      # peticiones en paralelo; bájalo si el proveedor limita la tasa
MAX_CARACTERES = 1500          # se recorta cada reseña (el p95 es ~485)
PILOTO = 100
EJECUTAR_TODO = False          # ← ponlo en True después de revisar el piloto
MAX_RESENAS = None             # tope opcional por presupuesto (None = las ~151 mil)
PRECIO_ENTRADA_USD_1M = None   # opcional, para estimar costo (USD por millón de tokens)
PRECIO_SALIDA_USD_1M = None
SEMILLA = 42

API key de JEV: ··········


In [13]:
import os

# Check if the folder exists
print(os.path.exists('/content/drive/MyDrive/TT_Recomendador_Cine_Mexicano/TT_Recomendador_Cine_Mexicano/datos/limpios/resenas_sentimiento.parquet'))

# List contents of the project folder to see actual paths
!ls -la "/content/drive/MyDrive/TT_Recomendador_Cine_Mexicano"

True
total 24
drwx------ 6 root root 4096 Nov  9 04:02 .
drwx------ 3 root root 4096 Nov  9 04:19 ..
drwx------ 2 root root 4096 Nov  9 04:02 datos
drwx------ 2 root root 4096 Nov  9 03:17 .git
drwx------ 2 root root 4096 Nov  9 04:02 resultados
drwx------ 9 root root 4096 Nov  9 04:17 TT_Recomendador_Cine_Mexicano


In [5]:
sys.path.append('/content/drive/MyDrive/TT_Recomendador_Cine_Mexicano/TT_Recomendador_Cine_Mexicano/src')

In [6]:
import json, re, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from recomendador import rutas, sentimiento as S

SALIDA = rutas.carpeta_resultados('exp5')
rutas.SENTIMIENTO.mkdir(parents=True, exist_ok=True)
CACHE = rutas.SENTIMIENTO / 'jev_cache.jsonl'
res = pd.read_parquet(rutas.LIMPIOS / 'resenas_sentimiento.parquet').sort_values('id').reset_index(drop=True)
print(f'{len(res):,} reseñas en español  ·  por grupo: {res.grupo.value_counts().to_dict()}')

151,516 reseñas en español  ·  por grupo: {'g5': 119872, 'g34': 31644}


## 1. La única función que depende del proveedor

Usa el formato *chat completions* compatible con OpenAI, que es el que ofrecen la mayoría de los servicios de LLM. **Si JEV usa otro formato, solo hay que cambiar esta celda**: recibe la lista de mensajes y devuelve el texto de la respuesta y el uso de tokens.

In [7]:
_local = threading.local()   # una sesión HTTP por hilo

def _sesion():
    if not hasattr(_local, 'sesion'):
        _local.sesion = requests.Session()
    return _local.sesion

def llamar_jev(mensajes, max_tokens=1200, intentos=6):
    """Envía `mensajes` a JEV y devuelve (texto_respuesta, uso_de_tokens)."""
    for intento in range(intentos):
        try:
            r = _sesion().post(
                f'{JEV_BASE_URL.rstrip("/")}/chat/completions',
                headers={'Authorization': f'Bearer {JEV_API_KEY}', 'Content-Type': 'application/json'},
                json={'model': JEV_MODELO, 'messages': mensajes, 'temperature': 0, 'max_tokens': max_tokens},
                timeout=120,
            )
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f'HTTP {r.status_code}', response=r)
            r.raise_for_status()
            datos = r.json()
            return datos['choices'][0]['message']['content'], datos.get('usage', {})
        except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as e:
            espera = min(60, 2 ** intento) + np.random.rand()
            if intento == intentos - 1:
                raise
            print(f'  reintento {intento + 1} en {espera:.0f} s ({e})')
            time.sleep(espera)

# Prueba de conexión
texto, uso = llamar_jev([{'role': 'user', 'content': 'Responde solo con la palabra: listo'}], max_tokens=5)
print('Respuesta:', texto, '· uso:', uso)

Respuesta: listo · uso: {'prompt_tokens': 18474, 'completion_tokens': 6, 'total_tokens': 18480}


## 2. Instrucciones (prompt)

Se pide la opinión del autor **sobre la película**, no sobre los personajes o la trama: "el villano es repugnante" puede venir de alguien a quien le encantó la película. Los ejemplos fijan el formato y calibran la escala.

In [8]:
INSTRUCCIONES = """Eres un anotador experto en análisis de sentimiento de reseñas de cine escritas en español, \
la mayoría de usuarios de México. Para cada reseña determina la opinión del AUTOR sobre la PELÍCULA \
(no sobre los personajes, la trama o los actores por separado):
- "positivo": le gustó o la recomienda.
- "neutral": opinión mixta, indiferente, o solo describe sin valorar.
- "negativo": no le gustó.
Además da una "puntuacion" continua entre -1.0 (muy negativa) y 1.0 (muy positiva), coherente con la etiqueta.
Ten en cuenta el sarcasmo, los modismos mexicanos ("está chida", "qué hueva", "un churro") y los emojis.

Responde ÚNICAMENTE con un arreglo JSON, sin texto antes ni después, con un objeto por reseña y en el mismo orden:
[{"id": 1, "sentimiento": "positivo", "puntuacion": 0.8}, ...]

Ejemplos:
[1] Qué chulada de película, lloré todo el final 😭
[2] Está bien para pasar el rato, nada memorable.
[3] Un churro, me dormí a la mitad.
[{"id": 1, "sentimiento": "positivo", "puntuacion": 0.9}, {"id": 2, "sentimiento": "neutral", "puntuacion": 0.05}, {"id": 3, "sentimiento": "negativo", "puntuacion": -0.85}]"""

def armar_mensajes(textos):
    cuerpo = '\n'.join(f'[{i}] {t[:MAX_CARACTERES]}' for i, t in enumerate(textos, 1))
    return [{'role': 'system', 'content': INSTRUCCIONES},
            {'role': 'user', 'content': f'Reseñas:\n{cuerpo}'}]

ETIQUETAS = {'positivo': 'positivo', 'positiva': 'positivo', 'neutral': 'neutral', 'neutro': 'neutral',
             'negativo': 'negativo', 'negativa': 'negativo', 'mixto': 'neutral', 'mixta': 'neutral'}

def interpretar(texto, n):
    """Extrae el arreglo JSON de la respuesta; devuelve {posición: (etiqueta, puntaje)}."""
    texto = re.sub(r'^```(?:json)?|```$', '', texto.strip(), flags=re.MULTILINE)
    ini, fin = texto.find('['), texto.rfind(']')
    if ini < 0 or fin < 0:
        return {}
    try:
        datos = json.loads(texto[ini:fin + 1])
    except json.JSONDecodeError:
        return {}
    salida = {}
    for d in datos if isinstance(datos, list) else []:
        try:
            i = int(d['id'])
            etq = ETIQUETAS.get(str(d.get('sentimiento', '')).strip().lower())
            s = float(d.get('puntuacion'))
        except (KeyError, TypeError, ValueError):
            continue
        if 1 <= i <= n and etq is not None and np.isfinite(s):
            salida[i] = (etq, float(np.clip(s, -1, 1)))
    return salida

print(armar_mensajes(['Muy buena 🔥', 'No me gustó'])[1]['content'])

Reseñas:
[1] Muy buena 🔥
[2] No me gustó


## 3. Procesamiento por lotes con caché

In [9]:
_candado = threading.Lock()
USO = {'prompt_tokens': 0, 'completion_tokens': 0, 'peticiones': 0, 'fallidas': 0, 'resenas': 0}  # solo esta sesión

def leer_cache():
    if not CACHE.exists():
        return {}
    hechos = {}
    with open(CACHE, encoding='utf-8') as f:
        for linea in f:
            d = json.loads(linea)
            hechos[d['id']] = d
    return hechos

def procesar_lote(lote: pd.DataFrame):
    textos = lote.texto_llm.tolist()
    try:
        respuesta, uso = llamar_jev(armar_mensajes(textos), max_tokens=40 * len(textos) + 100)
        resultado = interpretar(respuesta, len(textos))
    except Exception as e:
        with _candado:
            USO['fallidas'] += 1
        print('  lote fallido:', str(e)[:200])
        return 0
    # Lo que el modelo omitió en el lote se reintenta una por una
    for pos in set(range(1, len(textos) + 1)) - set(resultado):
        try:
            r1, u1 = llamar_jev(armar_mensajes([textos[pos - 1]]), max_tokens=80)
            uno = interpretar(r1, 1)
            if 1 in uno:
                resultado[pos] = uno[1]
            for k in ('prompt_tokens', 'completion_tokens'):
                uso[k] = uso.get(k, 0) + u1.get(k, 0)
        except Exception:
            pass
    filas = [{'id': rid, 'etiqueta': resultado[pos][0], 's': resultado[pos][1], 'modelo': JEV_MODELO}
             for pos, rid in enumerate(lote.id, 1) if pos in resultado]
    with _candado:
        with open(CACHE, 'a', encoding='utf-8') as f:
            for fila in filas:
                f.write(json.dumps(fila, ensure_ascii=False) + '\n')
        USO['peticiones'] += 1
        USO['resenas'] += len(filas)
        for k in ('prompt_tokens', 'completion_tokens'):
            USO[k] += int(uso.get(k, 0) or 0)
    return len(filas)

def procesar(df: pd.DataFrame, descripcion='reseñas'):
    hechos = leer_cache()
    pendientes = df[~df.id.isin(hechos)]
    print(f'{len(df) - len(pendientes):,} ya en caché · {len(pendientes):,} por procesar')
    lotes = [pendientes.iloc[i:i + RESENAS_POR_PETICION] for i in range(0, len(pendientes), RESENAS_POR_PETICION)]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=HILOS) as ex:
        futuros = [ex.submit(procesar_lote, l) for l in lotes]
        for _ in tqdm(as_completed(futuros), total=len(futuros), desc=descripcion):
            pass
    return time.time() - t0

def costo(uso):
    if PRECIO_ENTRADA_USD_1M is None or PRECIO_SALIDA_USD_1M is None:
        return None
    return uso['prompt_tokens'] / 1e6 * PRECIO_ENTRADA_USD_1M + uso['completion_tokens'] / 1e6 * PRECIO_SALIDA_USD_1M

## 4. Piloto (100 reseñas con estrellas)

In [10]:
piloto = res[res.calificacion.notna()].sample(PILOTO, random_state=SEMILLA)
segundos = procesar(piloto, 'piloto')
cache = pd.DataFrame(leer_cache().values())
assert len(cache), 'JEV no devolvió ninguna respuesta válida: revisa JEV_BASE_URL, JEV_MODELO y la API key.'
eval_piloto = piloto.merge(cache, on='id')
print(f'Respondidas: {len(eval_piloto)}/{len(piloto)} · peticiones fallidas: {USO["fallidas"]} · {segundos:.0f} s')
print('Tokens:', {k: USO[k] for k in ('prompt_tokens', 'completion_tokens', 'peticiones')})
if USO['resenas']:
    n = USO['resenas']
    por_resena = (USO['prompt_tokens'] + USO['completion_tokens']) / n
    total = len(res) if MAX_RESENAS is None else min(MAX_RESENAS, len(res))
    print(f'≈ {por_resena:.0f} tokens por reseña → ≈ {por_resena * total / 1e6:.1f} M tokens para {total:,} reseñas')
    c = costo(USO)
    if c is not None:
        print(f'Costo del piloto ≈ USD {c:.3f} → total estimado ≈ USD {c / n * total:,.2f}')
    print(f'Tiempo estimado total con {HILOS} hilos ≈ {segundos / n * total / 3600:.1f} h')
else:
    print('El piloto ya estaba en caché: no hay consumo nuevo que medir.')
resumen_piloto, matriz_piloto = S.evaluar_contra_estrellas(eval_piloto)
display(pd.Series(resumen_piloto).to_frame('piloto'))
display(eval_piloto[['calificacion', 'etiqueta', 's', 'texto_llm']].head(10))

0 ya en caché · 100 por procesar


piloto:   0%|          | 0/5 [00:00<?, ?it/s]

  reintento 1 en 2 s (HTTP 500)
  reintento 2 en 2 s (HTTP 500)
Respondidas: 100/100 · peticiones fallidas: 0 · 20 s
Tokens: {'prompt_tokens': 98454, 'completion_tokens': 3730, 'peticiones': 5}
≈ 1022 tokens por reseña → ≈ 154.8 M tokens para 151,516 reseñas
Tiempo estimado total con 4 hilos ≈ 8.4 h


,piloto
resenas_evaluadas,100
exactitud,0.55
f1_macro,0.511183
kappa_cohen,0.25546
f1_negativo,0.545455
f1_neutral,0.321429
f1_positivo,0.666667
spearman_s_vs_estrellas,0.538539
pearson_s_vs_estrellas,0.625599
mae_estrellas,0.758


,calificacion,etiqueta,s,texto_llm
0,3.5,positivo,0.65,"Entretenido y curiosamente grande, que si te a..."
1,4.5,neutral,0.00,"- ""No es fácil, es grande tarea cambiar lo que..."
2,4.5,positivo,0.80,"Ver ""Sin Señas Particulares"" es impactante no ..."
3,3.0,positivo,0.75,Sales de ver una película como Chronic y estás...
4,2.5,neutral,0.00,la verdad yo no me casaba con Tere ni por todo...
5,3.5,neutral,0.10,"Esta perla nos hará libres La verdad, no compr..."
6,4.0,neutral,0.00,"Si Possession es acerca de la guerra fría, el ..."
7,3.5,positivo,0.70,Me puse a ver de casualidad esta película de B...
8,2.5,negativo,-0.80,Vaya decepción con esta película. Y no es que ...
9,3.5,negativo,-0.30,"Pinche México, ¿acaso llegará el día en el que..."


## 5. Todas las reseñas

Revisa el piloto (calidad, tokens, costo). Si todo está bien, pon `EJECUTAR_TODO = True` en la celda de parámetros y vuelve a correr desde aquí. Con `MAX_RESENAS` puedes limitar el gasto: en ese caso se toma primero **todo el grupo 3–4** (31.6 mil reseñas) y luego una muestra aleatoria del grupo ≥ 5. Las reseñas que no se procesen simplemente no tendrán sentimiento (los experimentos usan solo sus estrellas).

In [ ]:
if EJECUTAR_TODO:
    objetivo = res
    if MAX_RESENAS is not None and MAX_RESENAS < len(res):
        g34 = res[res.grupo == 'g34']
        resto = res[res.grupo != 'g34'].sample(frac=1, random_state=SEMILLA)
        objetivo = pd.concat([g34, resto]).head(MAX_RESENAS)
    segundos = procesar(objetivo, 'todas')
    print(f'{segundos / 3600:.2f} h · tokens: {USO}')
else:
    print('EJECUTAR_TODO = False: solo se procesó el piloto.')

## 6. Archivo de salida y evaluación contra las estrellas

In [ ]:
cache = pd.DataFrame(leer_cache().values()).drop_duplicates('id', keep='last')
salida = res[['id', 'usuario', 'item', 'grupo', 'calificacion']].merge(cache, on='id', how='inner')
salida = S.validar(salida)
cobertura = len(salida) / len(res)
print(f'Reseñas con sentimiento JEV: {len(salida):,} ({cobertura:.1%} del total)')

if EJECUTAR_TODO:
    salida.to_parquet(rutas.ANALIZADORES['jev'], index=False)
    print('Guardado:', rutas.ANALIZADORES['jev'])
else:
    salida.to_parquet(rutas.SENTIMIENTO / 'sentimiento_jev_piloto.parquet', index=False)
    print('Piloto guardado aparte (no alimenta a los experimentos 1–3).')

resumen, matriz = S.evaluar_contra_estrellas(salida)
por_grupo = {g: S.evaluar_contra_estrellas(d)[0] for g, d in salida.groupby('grupo') if len(d) > 20}
tabla = pd.DataFrame({'todas': resumen, **por_grupo})
claves = ['resenas_evaluadas', 'exactitud', 'f1_macro', 'kappa_cohen', 'f1_negativo', 'f1_neutral', 'f1_positivo',
          'spearman_s_vs_estrellas', 'mae_estrellas']
display(tabla.loc[claves])
display(matriz)
tabla.to_csv(SALIDA / 'metricas_sentimiento_jev.csv')
matriz.to_csv(SALIDA / 'matriz_confusion_jev.csv')
with open(SALIDA / 'metricas_sentimiento_jev.json', 'w') as f:
    json.dump({'modelo': JEV_MODELO, 'cobertura': cobertura, 'uso_tokens': USO, 'costo_usd': costo(USO), **resumen},
              f, indent=2, default=float, ensure_ascii=False)
S.graficar_evaluacion(salida, matriz, f'JEV ({JEV_MODELO}) vs. estrellas', SALIDA / 'evaluacion_jev.png');

## 7. Acuerdo con BETO (si ya existe su archivo)

In [ ]:
if rutas.ANALIZADORES['beto'].exists():
    beto = pd.read_parquet(rutas.ANALIZADORES['beto'])
    acuerdo = S.acuerdo_entre_analizadores(beto, salida, ('beto', 'jev'))
    display(pd.Series(acuerdo).to_frame('BETO vs JEV'))
    with open(SALIDA / 'acuerdo_beto_jev.json', 'w') as f:
        json.dump(acuerdo, f, indent=2, default=float)
else:
    print('Aún no hay archivo de BETO (notebook 04).')

## 8. Siguiente paso

1. Descarga `datos/limpios/sentimiento/sentimiento_jev.parquet` (y `resultados/exp5/`) a tu carpeta local, en la misma ruta.
2. Vuelve a ejecutar `01`, `02` y `03`: correrán la **variante C** automáticamente.
3. Ejecuta `06_comparacion_resultados.ipynb`.

> El caché `jev_cache.jsonl` contiene solo identificadores y etiquetas, no la API key. Aun así, no subas la key a ningún repositorio.